In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [48]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
answer = llm("Hey, what's up !")

In [5]:
print(answer)

Hey! Not much—I'm here and ready to help. What’s going on?


In [7]:
question = "I just discovered the course. Can I join now ?"
answer = llm(question) 

In [8]:
answer

'Yes — you can join now, **if the course is still open for enrollment**.\n\nIf you want, I can help you figure out:\n- whether registration is still available,\n- what materials you’ve missed,\n- and the best way to catch up quickly.\n\nIf you’re asking about a specific course, send me the course name or link and I’ll help you check.'

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [1]:
import requests

In [2]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [3]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

In [5]:
for course in courses_raw:
    course_url = f"""{url_prefix}{course['path']}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [6]:
len(documents)

1208

In [8]:
print(documents[0])

{'id': '0e38656cfb', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How do I submit homework?', 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}


In [9]:
from minsearch import Index

In [10]:
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

In [11]:
index.fit(documents)

In [19]:
question = "I just discovered the course. Can I join now ?"

search_results = index.search(
    question,
    boost_dict={"question": 4.0, "section": 2.0},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [20]:
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [21]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [16]:
search_all_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

In [18]:
[doc['course'] for doc in search_all_results]

['llm-zoomcamp',
 'machine-learning-zoomcamp',
 'mlops-zoomcamp',
 'data-engineering-zoomcamp',
 'data-engineering-zoomcamp']

In [23]:
results = index.search(
    question,
    filter_dict={"course": "mlops-zoomcamp"},
    num_results=5
)

In [25]:
[doc['question'] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [27]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
    

In [28]:
search_results = search(question)

In [38]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [33]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    
    return "\n".join(lines).strip() 

In [42]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [35]:
print(question)

I just discovered the course. Can I join now ?


In [36]:
print(search_results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '977bf7786c', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?', 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}, {'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate 

In [43]:
prompt = build_prompt(question, search_results)

In [45]:
print(prompt)

Question:
I just discovered the course. Can I join now ?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your projec

In [49]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [52]:
print(response.output[0])

ResponseOutputMessage(id='msg_016cffab326fd659006a2166512c8c819d9b6a9b4a9cdcee19', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open and participate with the live cohort.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')


In [53]:
print(response.output[0].content[0].text)

Yes — you can still join now.

If you want a certificate, you’ll need to submit your project while submissions are still open and participate with the live cohort.


In [51]:
print(response.output_text)

Yes — you can still join now.

If you want a certificate, you’ll need to submit your project while submissions are still open and participate with the live cohort.


In [54]:
print(response.usage)

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=37, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=371)


In [55]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

In [56]:
cost = (
    response.usage.input_tokens * input_price + 
    response.usage.output_tokens * output_price
)

In [57]:
print(cost)

0.00041700000000000005


In [58]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [59]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [60]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [61]:
print(response.output_text)

Yes, you can still join now. If you want to receive a certificate, though, you need to submit your project while submissions are still being accepted.


In [62]:
def llm(instructions, prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [63]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)

    return answer

In [66]:
answer = rag("Does this course focus on mlops concepts ?")
print(answer)

I don't know.
